# Lab04A_Parte02_IAENG

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
dados = pd.read_csv("C:/Users/abalb/Desktop/precoscasas02.csv")

In [ ]:
dados = dados[["Price", "Area", "Room"]]

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.describe().apply(lambda s: s.apply('{0:.1f}'.format))

In [ ]:
fig, ax = plt.subplots(1, 3,figsize=(40,18))
sns.set(font_scale = 3)
ax1 = sns.boxplot(data=dados, x= 'Price', ax = ax[0])
ax1.set_xlabel('Preço em Milhões', fontsize = 30)
ax1.set_title('Análise de Outliers dos Preços de Casas', fontsize = 40)
ax2 = sns.boxplot(data=dados, x= 'Area', ax=ax[1])
ax2.set_xlabel('Área', fontsize = 30)
ax2.set_title('Análise de Outliers das Áreas', fontsize = 40)
ax3 = sns.boxplot(data=dados, x= 'Room', ax=ax[2])
ax3.set_xlabel('Quartos', fontsize = 30)
ax3.set_title('Análise de Outliers de Quartos', fontsize = 40)
plt.show()

#

In [ ]:
dados.describe().apply(lambda s: s.apply('{0:.1f}'.format))

In [ ]:
# calcular o IQR
Q1 = dados['Price'].quantile(0.25)
Q3 = dados['Price'].quantile(0.75)
IQR = Q3 - Q1
print(IQR)

In [ ]:
dados.loc[(dados['Price'] < (Q1 - 1.5 *IQR)) | (dados['Price'] > (Q3 + 1.5 * IQR)),'Price']

In [ ]:
plt.figure(figsize = (40,18))
ax = sns.histplot(data = dados, x= 'Price')
ax.set_xlabel('Preço em Milhões', fontsize = 30)
ax.set_ylabel('Contagem', fontsize = 30)
ax.set_title('Análise do Outlier dos Preços das Casas', fontsize = 40)
plt.xticks(fontsize = 30)
plt.yticks(fontsize = 30)

In [ ]:
plt.figure(figsize=(40,18))
ax = sns.boxplot(data=dados, x= 'Price')
ax.set_xlabel('Preço em Milhões', fontsize = 30)
ax.set_title('Análise de Outliers dos Preços de Casas', fontsize = 40)
plt.xticks(fontsize = 30)
plt.yticks(fontsize = 30)

#

In [ ]:
plt.figure(figsize = (40,18))
ax = sns.scatterplot(data= dados, x= 'Price', y ='Area', s = 200)
ax.set_xlabel('Preços', fontsize = 30)
ax.set_ylabel('Área', fontsize = 30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)
ax.set_title('Análise Outlier Bivariado dos Preços das Casas & da Área', fontsize = 40)

In [ ]:
plt.figure(figsize = (40,18))
ax = sns.boxplot(data= dados,x = 'Room', y= 'Price')
ax.set_xlabel('Quartos', fontsize = 30)
ax.set_ylabel('Preços', fontsize = 30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)
ax.set_title('Análise Outlier Bivariado dos Preços das Casas & da Área', fontsize = 40)

#

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import mahalanobis
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

In [ ]:
dados = pd.read_csv("C:/Users/abalb/Desktop/precoscasas02.csv")

In [ ]:
dados = dados[['Price', 'Area', 'Room', 'Lon', 'Lat']]

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
normalizacao = StandardScaler()
dados_normalizados = normalizacao.fit_transform(dados)
dados_normalizados = pd.DataFrame(dados_normalizados, columns=dados.columns)
dados_normalizados.head()

In [ ]:
media = dados_normalizados.mean()
cov = dados_normalizados.cov()
inv_cov = np.linalg.inv(cov)
distancias = []
for _, x in dados_normalizados.iterrows():
    d = mahalanobis(x, media, inv_cov)
    distancias.append(d)
dados['Distâncias de Mahalanobis'] = distancias
dados

In [ ]:
plt.figure(figsize = (40,18))
ax = sns.boxplot(data= dados, x= 'Distâncias de Mahalanobis')
ax.set_xlabel('Distâncias Mahalanobis', fontsize = 30)
plt.xticks(fontsize=30)
ax.set_title('Análise Outlier Analysis das Distâncias de Mahalanobis', fontsize = 40)

In [ ]:
outliers_mahalanobis = dados[dados['Distâncias de Mahalanobis']>=4]
outliers_mahalanobis.head()

In [ ]:
dados_sem_valores_ausentes = dados_normalizados.dropna()

In [ ]:
knn = NearestNeighbors(n_neighbors=2)
nbrs = knn.fit(dados_sem_valores_ausentes)
distancias, indices = nbrs.kneighbors(dados_sem_valores_ausentes)
distancias = np.sort(distancias, axis=0)
distancias = distancias[:,1]
plt.figure(figsize=(40,18))
plt.plot(distancias)
plt.title('Gráfico da distância-k',fontsize=40)
plt.xlabel('Dados classificados pela distância',fontsize=30)
plt.ylabel('Epsilon',fontsize=30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)
plt.show()

In [ ]:
dbscan = DBSCAN(eps= 1, min_samples=6)
dbscan.fit(dados_sem_valores_ausentes)
rotulos = dbscan.labels_
n_clusters = len(set(rotulos))
indices_outlier = np.where(rotulos == -1)[0]
print(f"Número de Clusters: {n_clusters}")
print(f"Número de Outliers: {len(indices_outlier)}")

In [ ]:
dados.loc[indices_outlier,:].head()

#

In [ ]:
dados = pd.read_csv("C:/Users/abalb/Desktop/precoscasas02.csv")

In [ ]:
dados = dados[['Price', 'Area', 'Room']]

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
Q1 = dados['Price'].quantile(0.25)
Q3 = dados['Price'].quantile(0.75)
IQR = Q3 - Q1
print(IQR)

In [ ]:
outliers = dados['Price'][(dados['Price'] < (Q1 - 1.5 * IQR)) |(dados['Price'] > (Q3 + 1.5 *IQR))]

In [ ]:
outliers.shape

In [ ]:
dados_sem_outliers = dados.drop(outliers.index, axis = 0)

In [ ]:
dados_sem_outliers.shape

In [ ]:
plt.figure(figsize = (40,18))
ax = sns.histplot(data= dados_sem_outliers, x= 'Price')
ax.set_xlabel('Preços em Milhões', fontsize = 30)
ax.set_title('Preços das Casas SEM Outliers', fontsize = 40)
ax.set_ylabel('Contagem', fontsize = 30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)

#

In [ ]:
substituicao_inferior = dados['Price'].quantile(0.05)
substituicao_superior = dados['Price'].quantile(0.95)
substuituicao_media = dados['Price'].quantile(0.5)
print(substituicao_inferior, substituicao_superior, substuituicao_media)

In [ ]:
dados['Adjusted_Price'] = dados.loc[:,'Price']
dados.loc[dados['Price'] < (Q1 - 1.5 * IQR),'Adjusted_Price'] = substituicao_inferior
dados.loc[dados['Price'] > (Q3 + 1.5 * IQR), 'Adjusted_Price'] = substituicao_superior

In [ ]:
outliers = dados['Price'][(dados['Price'] < (Q1 - 1.5 * IQR)) | (dados['Price'] > (Q3 + 1.5 * IQR))]
dados_substituidos = dados.copy()
dados_substituidos.loc[:,'Price_with_nan'] = dados_substituidos.loc[:,'Price']
dados_substituidos.loc[outliers.index,'Price_with_nan'] = np.nan
dados_substituidos.isnull().sum()

In [ ]:
dados_substituidos = dados_substituidos.interpolate(method='linear')
dados_substituidos.loc[outliers.index,:].head()

In [ ]:
plt.figure(figsize = (40,18))
ax = sns.histplot(data= dados_substituidos, x= 'Price_with_nan')
ax.set_xlabel('Preços das Casas Interpolados em Milhões', fontsize = 30)
ax.set_title('Preços das casas com substituição dos outliers', fontsize = 40)
ax.set_ylabel('Contagem', fontsize = 30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)

#

In [ ]:
# inicialmente instalar o pacote
# deixo como forte sugestão a pesquisa sobre este pacote
# !pip3 install missingno

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

In [ ]:
dados = pd.read_csv("C:/Users/abalb/Desktop/precoscasas02.csv")

In [ ]:
dados = dados[['Price', 'Area', 'Room']]

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.isnull().sum()

In [ ]:
dados.info()

In [ ]:
msno.bar(dados)

In [ ]:
msno.matrix(dados)

In [ ]:
dados[dados['Price'].isnull()]

#

In [ ]:
remocao_dados_ausentes = dados.dropna()

In [ ]:
remocao_dados_ausentes.shape

In [ ]:
remocao_dados_ausentes_subconjunto = dados.dropna(subset = ['Price', 'Area'])

In [ ]:
remocao_dados_ausentes_subconjunto.shape

In [ ]:
remocao_coluna_dados = dados.drop(['Area'],axis= 1)

In [ ]:
remocao_coluna_dados.shape

#

In [ ]:
dados = pd.read_csv("C:/Users/abalb/Desktop/precoscasas02.csv")

In [ ]:
dados = dados[['Zip', 'Price', 'Area','Room']]

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.isnull().sum()

In [ ]:
dados[dados['Price'].isnull()]

In [ ]:
media = dados['Price'].mean()
mediana = dados['Price'].median()
moda = dados['Zip'].mode()[0]
print("média: ",media,"mediana: " ,mediana,"moda: ", moda)

In [ ]:
dados['Média dos Preços'] = dados['Price'].fillna(media)
dados.isnull().sum()

In [ ]:
dados['Mediana dos Preços'] = dados['Price'].fillna(mediana)
dados.isnull().sum()

In [ ]:
dados[dados['Price'].isnull()]

#

In [ ]:
import pandas as pd
from sklearn.impute import KNNImputer

In [ ]:
dados = pd.read_csv("C:/Users/abalb/Desktop/precoscasas02.csv")

In [ ]:
dados = dados[['Price', 'Area','Room','Lon','Lat']]

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.isnull().sum()

In [ ]:
indice_dos_dados_ausentes = dados[dados['Price'].isnull()].index
indice_dos_dados_ausentes

In [ ]:
imputacao = KNNImputer(n_neighbors=5)
dados_imputados_knn = pd.DataFrame(imputacao.fit_transform(dados),columns = dados.columns)

In [ ]:
dados_imputados_knn.loc[indice_dos_dados_ausentes,:]